# BaseDataset and collate_fn

This tutorial shows how to implement a minimal `BaseDataset` subclass and use a custom `collate_fn` with PyTorch's `DataLoader`. The flow is: **container (data_dict) → Dataset → batch**.

## Data flow: container → dataset → batch

```mermaid
flowchart LR
    A[data_dict] --> B[Dataset]
    B --> C[__getitem__]
    C --> D[list of samples]
    D --> E[collate_fn]
    E --> F[batch]
```

- **data_dict**: Dict with keys like `"features"`, `"target"` mapping to arrays of shape `(N, D)` and `(N,)`.
- **Dataset**: `__getitem__(idx)` returns a single sample dict: `{"features": tensor, "target": tensor}`.
- **collate_fn**: Combines a list of samples into a batch by stacking tensors along the batch dimension.

## Role of collate_fn

The `collate_fn` is passed to `DataLoader` and receives a list of samples (one per index in the batch). It must return a single dict that stacks tensors so the batch has shape `(batch_size, ...)` for each key. For simple tabular data, `torch.stack` is the standard choice. The project also provides `collate_key_value_batch` (from `picid.data.datasets.collate_functions`) for dicts with tensor values; for 1D tensors per sample, a custom stack-based collate is often clearer.

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader

from picid.data.datasets.base import BaseDataset


class SimpleTabularDataset(BaseDataset):
    """Minimal dataset extending BaseDataset for features (N, D) and target (N,)."""

    def __init__(self, data_dict):
        super().__init__(data_dict)
        self.features = np.asarray(data_dict["features"])
        self.target = np.asarray(data_dict["target"])
        assert len(self.features) == len(
            self.target
        ), "features and target must have same length"

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return {
            "features": torch.as_tensor(self.features[idx], dtype=torch.float32),
            "target": torch.as_tensor(self.target[idx], dtype=torch.float32),
        }

    def get_collate_fn(self):
        def collate(batch):
            return {
                "features": torch.stack([b["features"] for b in batch]),
                "target": torch.stack([b["target"] for b in batch]),
            }

        return collate

In [ ]:
N, D = 12, 4
data_dict = {
    "features": np.random.randn(N, D).astype(np.float32),
    "target": np.random.randn(N).astype(np.float32),
}
dataset = SimpleTabularDataset(data_dict)
loader = DataLoader(
    dataset,
    batch_size=4,
    collate_fn=dataset.get_collate_fn(),
)
batch = next(iter(loader))
print(f"Batch features shape: {batch['features'].shape}")
print(f"Batch target shape: {batch['target'].shape}")
assert batch["features"].shape[0] == 4
assert batch["target"].shape[0] == 4
print("OK")